# From Policy Gradient to Actor-Critic: An Interactive Deep RL Walkthrough (CartPole)

Reinforcement learning (RL) is just the **same three-step machine-learning recipe** you already know — *define a function, define a loss, optimize* — applied to an agent that learns by trial and error. In this notebook we build that intuition step by step on the classic **CartPole** balancing task.

## The narrative spine (five concepts)

1. **RL as "looking for a function"** — the policy network maps `Action = f(Observation)`, and why actions are *sampled* (exploration) instead of taken greedily.
2. **Policy gradient as weighted classification** — and the version progression of the weighting term: `V0` immediate reward → `V1` cumulated return → `V2` discounted return.
3. **Baseline & advantage normalization** — subtracting a baseline so "good or bad" becomes *relative* (`V3`).
4. **The critic: Monte-Carlo vs Temporal-Difference** — estimating value, illustrated with Sutton's classic 8-episode example.
5. **Advantage Actor-Critic (A2C)** — `V4` advantage `A_t = r_t + V(s_{t+1}) - V(s_t)`, the culmination of the progression.

## Learning objectives

- Frame RL as the three-step ML recipe where `Action = f(Observation)` via a policy network.
- Build a policy network for CartPole and see why actions are sampled rather than taken greedily.
- Derive the policy-gradient weighting term and watch the advantage evolve V0 → V1 → V2.
- Understand how subtracting a baseline and normalizing turns absolute returns into relative advantages (V3).
- Contrast Monte-Carlo and Temporal-Difference value estimation using the Sutton 8-episode example.
- Assemble an A2C agent and compare its learning curve against baseline-free policy gradient.

> **How to run:** Execute the cells top-to-bottom. Cells marked *interactive* expose sliders/toggles when `ipywidgets` is available, and fall back to a static plot otherwise so the notebook always runs headless.

In [ ]:
# C02 — Environment setup: installs, imports, seeding, and shared helpers.
%matplotlib inline
import sys, subprocess, random

# (1) Quietly install gymnasium / matplotlib only if importing them fails.
try:
    import gymnasium as gym
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium'], check=False)
    import gymnasium as gym

try:
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'], check=False)
    import matplotlib

# (2) Core scientific / DL imports.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ipywidgets is optional: degrade gracefully if unavailable (headless verification).
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

# (3) Reproducibility.
SEED = 0
DEVICE = torch.device('cpu')

def set_seed(seed: int = 0) -> None:
    """Seed Python, numpy, and torch RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

# (4) Smoothing helper used by the learning-curve cells.
def moving_average(x, window: int = 10) -> np.ndarray:
    """Return a 1-D moving average; window is clamped to [1, len(x)]."""
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    window = max(1, min(int(window), len(x)))
    return np.convolve(x, np.ones(window) / window, mode='valid')

# (5) Consistent figure size.
plt.rcParams['figure.figsize'] = (7, 4)

# (6) Confirm the runtime the reader is on.
try:
    _gym_ver = gym.__version__
except Exception:
    _gym_ver = 'unknown'
print(f"torch={torch.__version__}  numpy={np.__version__}  gymnasium={_gym_ver}")
print(f"DEVICE={DEVICE}  WIDGETS_AVAILABLE={WIDGETS_AVAILABLE}")

# --- self-checks ---
assert WIDGETS_AVAILABLE in (True, False)
assert callable(set_seed) and callable(moving_average)
assert len(moving_average(np.arange(10), 3)) == 8
assert str(DEVICE) == 'cpu'
print('C02 setup OK')

## Concept 1 — Reinforcement Learning as "looking for a function"

RL fits the **same three-step recipe** as supervised learning:

| Step | Supervised learning | Reinforcement learning |
|------|--------------------|------------------------|
| 1. Function | `y = f(x)` | **`action = f(observation)`** (the *policy* / *actor*) |
| 2. Loss | cross-entropy, MSE, … | **maximize total reward** `R = Σ r_t` |
| 3. Optimization | gradient descent | gradient *ascent* on expected reward |

The **actor** is a neural network (a *policy network*) that takes the current observation and outputs a **softmax distribution over actions**. We play one **episode** — a full sequence of (observation → action → reward) until the pole falls or time runs out — and the quantity we want to maximize is the **return** `R = Σ_t r_t`.

### Why this is harder than supervised learning

The environment is a **non-differentiable black box**: we cannot backpropagate through `env.step()`. There are no ground-truth labels saying "the correct action here was X." Instead, the only signal is the reward we collect, which may arrive **long after** the action that caused it (the *credit-assignment* problem). The rest of the notebook is about turning that delayed, noisy reward signal into a usable training gradient.

In [ ]:
# C04 — Instantiate CartPole-v1 and inspect its interface.
env = gym.make('CartPole-v1')

obs_dim = env.observation_space.shape[0]   # expect 4 continuous values
n_actions = env.action_space.n             # expect 2 discrete actions

# Reset with the seeded gymnasium API (fall back for very old gym).
try:
    obs, info = env.reset(seed=SEED)
except TypeError:
    obs = env.reset()
    info = {}

# Take one random step to show the loop and the 5-tuple step API.
a = env.action_space.sample()
step_result = env.step(a)
if len(step_result) == 5:
    obs2, reward, terminated, truncated, info = step_result
else:  # legacy 4-tuple gym: (obs, reward, done, info)
    obs2, reward, done, info = step_result
    terminated, truncated = bool(done), False

print('=== CartPole-v1 interface ===')
print(f'observation space : {env.observation_space.shape}  ->  obs_dim = {obs_dim}')
print(f'sample observation: {np.round(obs, 4)}')
print(f'action space      : {env.action_space}  ->  n_actions = {n_actions}')
print(f'one random step   : action={a}, reward={reward}, terminated={terminated}, truncated={truncated}')

# --- self-checks ---
assert obs_dim == 4
assert n_actions == 2
assert env.observation_space.contains(obs)
assert isinstance(reward, (int, float))
print('C04 env inspection OK')

In [ ]:
# C05 — Define the actor (PolicyNetwork) and visualize its initial action distribution.
class PolicyNetwork(nn.Module):
    """Small MLP actor: observation -> action logits (softmax gives probabilities)."""
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)  # raw logits

    def action_probs(self, obs: torch.Tensor) -> torch.Tensor:
        return F.softmax(self.forward(obs), dim=-1)

# Reproducible weights.
set_seed(SEED)
policy = PolicyNetwork(obs_dim, n_actions)

# Forward pass on the reset observation.
obs_t = torch.as_tensor(np.asarray(obs, dtype=np.float32)).reshape(1, obs_dim)
with torch.no_grad():
    logits = policy(obs_t)
    assert torch.isfinite(logits).all(), 'non-finite logits'
    probs = F.softmax(logits, dim=-1).squeeze(0).numpy()

plt.figure()
plt.bar(range(n_actions), probs, color=['#4C72B0', '#DD8452'])
plt.xticks(range(n_actions), [f'action {i}' for i in range(n_actions)])
plt.ylabel('probability')
plt.ylim(0, 1)
plt.title('Untrained policy: action probabilities for the initial observation')
for i, p in enumerate(probs):
    plt.text(i, p + 0.02, f'{p:.2f}', ha='center')
plt.show()

print('action probabilities:', np.round(probs, 4))

# --- self-checks ---
assert probs.shape == (n_actions,)
assert abs(float(probs.sum()) - 1.0) < 1e-5
assert (probs >= 0).all()
print('C05 policy network OK')

In [ ]:
# C06 — Exploration: greedy (argmax) vs stochastic (sampled) action selection.
from torch.distributions import Categorical

def rollout_policy(policy: PolicyNetwork, env, mode: str = 'sample', max_steps: int = 500) -> float:
    """Run one episode under the given selection mode; return the total (undiscounted) return."""
    if mode not in ('greedy', 'sample'):
        raise ValueError(f"mode must be 'greedy' or 'sample', got {mode!r}")
    obs, _ = env.reset()
    total = 0.0
    with torch.no_grad():
        for _ in range(max_steps):
            obs_t = torch.as_tensor(np.asarray(obs, dtype=np.float32)).reshape(1, obs_dim)
            p = policy.action_probs(obs_t).squeeze(0)
            if mode == 'greedy':
                action = int(torch.argmax(p).item())
            else:
                action = int(Categorical(p).sample().item())
            obs, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            if terminated or truncated:
                break
    return total

def run_episodes(mode: str, n_episodes: int) -> list:
    return [rollout_policy(policy, env, mode) for _ in range(n_episodes)]

def draw(mode: str = 'sample', n_episodes: int = 20) -> None:
    """Overlay return histograms for the chosen mode vs the other mode and print stats."""
    other = 'sample' if mode == 'greedy' else 'greedy'
    r_sel = run_episodes(mode, n_episodes)
    r_oth = run_episodes(other, n_episodes)
    plt.figure()
    bins = np.linspace(0, max(max(r_sel), max(r_oth), 1), 15)
    plt.hist(r_sel, bins=bins, alpha=0.6, label=f'{mode} (selected)')
    plt.hist(r_oth, bins=bins, alpha=0.6, label=f'{other}')
    plt.xlabel('episode return')
    plt.ylabel('count')
    plt.title(f'Return distribution: {mode} vs {other} ({n_episodes} episodes each)')
    plt.legend()
    plt.show()
    print(f'{mode:>7}: mean={np.mean(r_sel):.1f}  std={np.std(r_sel):.1f}')
    print(f'{other:>7}: mean={np.mean(r_oth):.1f}  std={np.std(r_oth):.1f}')

if WIDGETS_AVAILABLE:
    widgets.interact(
        draw,
        mode=widgets.ToggleButtons(options=['greedy', 'sample'], value='sample'),
        n_episodes=widgets.IntSlider(min=5, max=50, step=5, value=20),
    )
else:
    print('[ipywidgets unavailable — static comparison]')
    draw('sample', 20)

# --- self-checks ---
assert all(r >= 0 for r in run_episodes('greedy', 3))
assert isinstance(rollout_policy(policy, env, 'sample'), float)
draw('sample', 5)
print('C06 exploration demo OK')

## Concept 2 — Policy gradient as *weighted classification*

If there were a "correct" action label for every state, training the actor would be ordinary classification with cross-entropy `e_n`. We don't have labels — but we *can* weight each action's cross-entropy term by **how good that action turned out to be**:

$$ L = \sum_n A_n \, e_n $$

Actions followed by high reward get pushed up; actions followed by low reward get pushed down. Everything hinges on **what we plug in for the weight `A_n`**, and the lecture builds it up in versions:

- **V0 — immediate reward `r_t`.** Weight each action by the reward received *on that step*. Short-sighted: an action's true value often shows up many steps later (in CartPole every step gives `+1`, so V0 carries almost no signal).
- **V1 — cumulated return `G_t = Σ_{k≥t} r_k`.** Credit an action with *all* future reward it helped earn. This fixes the reward-delay problem but treats reward 100 steps away as equally important as the next step.
- **V2 — discounted return `G'_t = Σ_{k≥t} γ^{k-t} r_k`, with `γ < 1`.** Future reward is geometrically discounted, so nearer consequences matter more — a tunable **credit horizon**.

The next cell computes V0, V1, and V2 on a real CartPole trajectory and lets you slide `γ` to watch the credit horizon stretch and shrink.

In [ ]:
# C08 — Collect one on-policy trajectory used by the return-computation demos.
from torch.distributions import Categorical

def collect_trajectory(policy: PolicyNetwork, env, max_steps: int = 500):
    """Roll out one stochastic episode; return (states, actions, rewards) numpy arrays."""
    states, actions, rewards = [], [], []
    obs, _ = env.reset()
    with torch.no_grad():
        for _ in range(max_steps):
            s = np.asarray(obs, dtype=np.float32)
            obs_t = torch.as_tensor(s).reshape(1, obs_dim)
            p = policy.action_probs(obs_t).squeeze(0)
            action = int(Categorical(p).sample().item())
            obs, reward, terminated, truncated, _ = env.step(action)
            states.append(s)
            actions.append(action)
            rewards.append(float(reward))
            if terminated or truncated:
                break
    if len(rewards) == 0:  # extremely unlikely immediate termination — retry once
        return collect_trajectory(policy, env, max_steps)
    return (
        np.asarray(states, dtype=np.float32),
        np.asarray(actions, dtype=np.int64),
        np.asarray(rewards, dtype=np.float32),
    )

set_seed(SEED)
traj_states, traj_actions, traj_rewards = collect_trajectory(policy, env)
print(f'trajectory length = {len(traj_rewards)} steps')
print(f'total undiscounted reward = {traj_rewards.sum():.1f}')

# --- self-checks ---
assert traj_states.shape[1] == obs_dim
assert len(traj_actions) == len(traj_rewards) == traj_states.shape[0]
assert traj_rewards.sum() > 0
print('C08 trajectory collection OK')

In [ ]:
# C09 — Compute & visualize the V0/V1/V2 weighting terms with an interactive gamma slider.
def compute_returns(rewards, gamma: float = 0.99, mode: str = 'v2') -> np.ndarray:
    """Per-timestep weighting term.
    v0: raw per-step reward r_t.
    v1: cumulated future reward (reverse cumsum, gamma=1).
    v2: discounted future return G'_t = sum_{k>=t} gamma^{k-t} r_k.
    """
    if mode not in ('v0', 'v1', 'v2'):
        raise ValueError(f"mode must be one of 'v0','v1','v2', got {mode!r}")
    if not (0.0 <= gamma <= 1.0):
        raise ValueError(f'gamma must be in [0,1], got {gamma}')
    r = np.asarray(rewards, dtype=np.float64)
    if r.size == 0:
        return r
    if mode == 'v0':
        return r.copy()
    g = 1.0 if mode == 'v1' else gamma
    out = np.zeros_like(r)
    running = 0.0
    for t in range(len(r) - 1, -1, -1):
        running = r[t] + g * running
        out[t] = running
    return out

def plot_weightings(gamma: float = 0.99) -> None:
    v0 = compute_returns(traj_rewards, mode='v0')
    v1 = compute_returns(traj_rewards, mode='v1')
    v2 = compute_returns(traj_rewards, gamma=gamma, mode='v2')
    plt.figure()
    plt.plot(v0, label='V0: immediate r_t')
    plt.plot(v1, label='V1: cumulated return (gamma=1)')
    plt.plot(v2, label=f'V2: discounted return (gamma={gamma:.2f})')
    plt.xlabel('timestep')
    plt.ylabel('weighting term A_t')
    plt.title('Policy-gradient weighting: V0 vs V1 vs V2')
    plt.legend()
    plt.show()
    print(f'gamma={gamma:.2f}: higher gamma lengthens the credit horizon (V2 -> V1 as gamma -> 1).')

if WIDGETS_AVAILABLE:
    widgets.interact(plot_weightings, gamma=widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.99))
else:
    print('[ipywidgets unavailable — static gamma=0.99 plot]')
    plot_weightings(0.99)

# --- self-checks ---
assert np.allclose(compute_returns(np.ones(5), gamma=1.0, mode='v2'), [5, 4, 3, 2, 1])
assert np.allclose(compute_returns(traj_rewards, mode='v0'), traj_rewards)
assert compute_returns(traj_rewards, gamma=0.99, mode='v2').shape == traj_rewards.shape
print('C09 return computation OK')

## Concept 3 — Reward baseline & advantage normalization (V3)

There is a pathology hiding in V2. In CartPole **every reward is `+1`**, so every discounted return `G'_t` is **positive**. With the loss `L = Σ A_n e_n` and all weights positive, the actor tries to make *every* action it ever took *more* likely — it can only distinguish actions by *how positive* they are, which is a weak, high-variance signal.

The fix: subtract a **baseline `b`** so the weight becomes an **advantage**:

$$ A_t = G'_t - b $$

Now `A_t` is **positive for better-than-average actions and negative for worse-than-average ones** — *"good or bad is relative."* Actions below the baseline are actively pushed *down*. A simple, effective baseline is the **mean return**, optionally followed by dividing by the standard deviation (**standardization**) to keep gradient magnitudes stable across episodes:

$$ \hat{A}_t = \frac{G'_t - \text{mean}(G')}{\text{std}(G') + \epsilon} $$

This is exactly the V3 trick. Later, the **critic** `V_θ(s)` will *learn* a state-dependent baseline (the expected return from each state), which is even better than a single global mean — the bridge to Actor-Critic.

In [ ]:
# C11 — V3 demo: baseline subtraction + standardization turns returns into zero-centered advantages.
def normalize_advantages(returns, eps: float = 1e-8) -> np.ndarray:
    """Subtract the mean (baseline) and divide by (std + eps): zero-centered, ~unit-variance."""
    r = np.asarray(returns, dtype=np.float64)
    return (r - r.mean()) / (r.std() + eps)

# Collect several trajectories and concatenate their discounted (V2) returns.
set_seed(SEED)
_all = []
for _ in range(8):
    _, _, rew = collect_trajectory(policy, env)
    _all.append(compute_returns(rew, gamma=0.99, mode='v2'))
raw_returns = np.concatenate(_all)
norm_adv = normalize_advantages(raw_returns)

def redraw(normalize: bool = True) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(raw_returns, bins=30, color='#4C72B0')
    axes[0].axvline(raw_returns.mean(), color='k', ls='--', label=f'mean={raw_returns.mean():.1f}')
    axes[0].set_title('Before: raw discounted returns (all positive)')
    axes[0].set_xlabel("G'_t"); axes[0].legend()
    axes[1].hist(norm_adv, bins=30, color='#DD8452')
    axes[1].axvline(0.0, color='k', ls='--', label='zero')
    axes[1].set_title('After: normalized advantage (+ and -)')
    axes[1].set_xlabel('A_t'); axes[1].legend()
    if normalize:
        axes[1].patch.set_alpha(1.0); axes[0].patch.set_alpha(0.4)
    else:
        axes[0].patch.set_alpha(1.0); axes[1].patch.set_alpha(0.4)
    plt.tight_layout()
    plt.show()
    print(f'raw   : mean={raw_returns.mean():.2f}  std={raw_returns.std():.2f}')
    print(f'normed: mean={norm_adv.mean():.2e}  std={norm_adv.std():.2f}')

if WIDGETS_AVAILABLE:
    widgets.interact(redraw, normalize=widgets.Checkbox(value=True, description='emphasize normalized'))
else:
    print('[ipywidgets unavailable — static before/after comparison]')
    redraw(True)

# --- self-checks ---
assert abs(float(norm_adv.mean())) < 1e-5
assert abs(float(norm_adv.std()) - 1.0) < 1e-3
assert raw_returns.shape == norm_adv.shape
print('C11 advantage normalization OK')

## Concept 4 — The critic: Monte-Carlo vs Temporal-Difference

The **critic** `V_θ(s)` estimates the **expected discounted return** starting from state `s` under the current actor. It is the *learned* baseline from Concept 3. How do we train it from experience? Two classic recipes:

- **Monte-Carlo (MC).** Wait until the episode ends, then set `V(s_t) ← G'_t` (the actual return that followed). *Unbiased* but **high variance** — it depends on the entire noisy rest of the episode.
- **Temporal-Difference (TD).** Bootstrap from your own next estimate: `V(s_t) ← r_t + γ V(s_{t+1})`. *Lower variance* (one step of real reward + a learned estimate) but **biased** while `V` is still wrong.

This is the **bias/variance trade-off** of value estimation, and **bootstrapping** (TD using `V` to estimate `V`) is what lets it exploit structure MC ignores.

### Sutton's 8-episode example (Example 6.4)

A tiny dataset makes the difference vivid. Two states, `A` and `B`. We observe 8 episodes; only **one** of them visits `A`, and in that episode `A → B` gives reward 0 and then the episode ends with reward 0. The other 7 episodes start at `B`. From this same data, MC and TD give **different** estimates of `V(A)` — the next cell computes both and explains why.

In [ ]:
# C13 — Sutton Example 6.4: MC vs TD estimate of V(A) on the canonical 8-episode batch.
# Each episode is a list of (state, reward) transitions.
# Episode 1: A -> reward 0, then B -> reward 0 (A's only appearance).
# Episodes 2-7: B -> reward 1.  Episode 8: B -> reward 0.
eight_episodes = [
    [('A', 0.0), ('B', 0.0)],
    [('B', 1.0)],
    [('B', 1.0)],
    [('B', 1.0)],
    [('B', 1.0)],
    [('B', 1.0)],
    [('B', 1.0)],
    [('B', 0.0)],
]

def _episode_return(ep):
    return sum(r for _, r in ep)

def mc_value(episodes, state: str) -> float:
    """Monte-Carlo: mean total return over episodes that visit `state`."""
    rets = [_episode_return(ep) for ep in episodes if any(s == state for s, _ in ep)]
    if not rets:
        print(f'state {state!r} never visited'); return float('nan')
    return float(np.mean(rets))

def td_value(episodes, state: str) -> float:
    """TD fixed point: V(state) = mean over its transitions of r + V(next).
    With V(B) = mean reward observed at B, V(A) = r(A->B) + V(B)."""
    b_rewards = [r for ep in episodes for (s, r) in ep if s == 'B']
    v_b = float(np.mean(b_rewards)) if b_rewards else float('nan')
    if state == 'B':
        return v_b
    vals = []
    for ep in episodes:
        for i, (s, r) in enumerate(ep):
            if s == state:
                nxt = ep[i + 1][0] if i + 1 < len(ep) else None
                vals.append(r + (v_b if nxt == 'B' else 0.0))
    if not vals:
        print(f'state {state!r} never visited'); return float('nan')
    return float(np.mean(vals))

v_b = td_value(eight_episodes, 'B')
mc_a = mc_value(eight_episodes, 'A')
td_a = td_value(eight_episodes, 'A')

print(f'V(B)              = {v_b:.3f}   (6 of 8 B-episodes give reward 1)')
print(f'V(A)  Monte-Carlo = {mc_a:.3f}   (A appeared once, that episode returned 0)')
print(f'V(A)  TD          = {td_a:.3f}   (A -> B gives 0 + V(B) = {v_b:.2f})')
print('\nMC sees only A\'s single zero-return episode; TD exploits the A->B transition structure.')

plt.figure()
plt.bar(['MC', 'TD'], [mc_a, td_a], color=['#4C72B0', '#DD8452'])
plt.ylabel('estimated V(A)')
plt.title('Sutton Example 6.4: MC vs TD estimate of V(A)')
for i, v in enumerate([mc_a, td_a]):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')
plt.show()

# --- self-checks ---
assert abs(v_b - 0.75) < 1e-9
assert abs(mc_a - 0.0) < 1e-9
assert abs(td_a - 0.75) < 1e-9
assert len(eight_episodes) == 8
print('C13 MC-vs-TD example OK')

## Concept 5 — Advantage Actor-Critic (A2C): the culmination (V4)

We now combine the actor (Concept 1–3) with the learned critic (Concept 4). Recall the advantage `A_t = G'_t - b`, where the *best* baseline `b` is the critic `V_θ(s_t)`. But `G'_t` is still a noisy Monte-Carlo sample. Using the **TD** view, we replace it with a one-step bootstrap:

$$ A_t = \underbrace{r_t + \gamma V_\theta(s_{t+1})}_{\text{TD target}} - V_\theta(s_t) $$

This is **Version 4** — the TD advantage. Compared to subtracting a baseline from the full sampled return `G`, it has **much lower variance**: only the single reward `r_t` is sampled, and the rest is summarized by the critic.

The two roles can **share parameters**: one network with a common trunk and two heads — a **policy head** (logits, the actor) and a **value head** (scalar `V`, the critic). This is the **Advantage Actor-Critic (A2C)** architecture, and it is the natural endpoint of the whole V0 → V1 → V2 → V3 → V4 progression we have been building. The next cells train it on CartPole and compare it against baseline-free policy gradient.

In [ ]:
# C15 — Define and train an Advantage Actor-Critic (A2C) agent on CartPole (timeout-safe budget).
from torch.distributions import Categorical

N_EPISODES = 250          # shared training budget (also used by C16) — kept small for the verifier
ENTROPY_COEF = 0.01

class ActorCritic(nn.Module):
    """Shared trunk with a policy head (logits) and a value head (scalar V)."""
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64) -> None:
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh())
        self.policy_head = nn.Linear(hidden, n_actions)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor):
        h = self.trunk(x)
        return self.policy_head(h), self.value_head(h)

def train_a2c(env, n_episodes: int = N_EPISODES, gamma: float = 0.99, lr: float = 3e-3):
    set_seed(SEED)
    model = ActorCritic(obs_dim, n_actions)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    returns_hist = []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        log_probs, values, rewards, entropies = [], [], [], []
        for _ in range(500):
            obs_t = torch.as_tensor(np.asarray(obs, dtype=np.float32)).reshape(1, obs_dim)
            logits, value = model(obs_t)
            dist = Categorical(logits=logits)
            action = dist.sample()
            obs, reward, terminated, truncated, _ = env.step(int(action.item()))
            log_probs.append(dist.log_prob(action).squeeze())
            values.append(value.squeeze())
            entropies.append(dist.entropy().squeeze())
            rewards.append(float(reward))
            if terminated or truncated:
                break
        returns_hist.append(float(np.sum(rewards)))
        G = torch.as_tensor(compute_returns(np.asarray(rewards), gamma=gamma, mode='v2'), dtype=torch.float32)
        values_t = torch.stack(values)
        log_probs_t = torch.stack(log_probs)
        entropy = torch.stack(entropies).mean()
        advantage = (G - values_t).detach()
        actor_loss = -(log_probs_t * advantage).sum()
        critic_loss = F.mse_loss(values_t, G)
        loss = actor_loss + 0.5 * critic_loss - ENTROPY_COEF * entropy
        if torch.isfinite(loss):
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep + 1) % 50 == 0:
            print(f'A2C ep {ep+1:>3}/{n_episodes}  return(avg last 20)={np.mean(returns_hist[-20:]):.1f}')
    return model, returns_hist

a2c_model, a2c_returns = train_a2c(env)
print(f'A2C done: first-20 avg={np.mean(a2c_returns[:20]):.1f}  last-20 avg={np.mean(a2c_returns[-20:]):.1f}')
if np.mean(a2c_returns[-20:]) < np.mean(a2c_returns[:20]):
    print('[warn] no upward learning trend this run — RL variance; trend is the qualitative takeaway')

# --- self-checks ---
assert isinstance(a2c_model, ActorCritic)
assert len(a2c_returns) == N_EPISODES
_lg, _v = a2c_model(torch.zeros(1, obs_dim))
assert _lg.shape == (1, n_actions) and _v.shape[-1] == 1
print('C15 A2C training OK')

In [ ]:
# C16 — Train baseline-free REINFORCE under the same budget and overlay vs A2C.
from torch.distributions import Categorical

def train_reinforce(env, n_episodes: int = N_EPISODES, gamma: float = 0.99, lr: float = 3e-3):
    """REINFORCE with discounted return and no learned critic baseline — the contrast point."""
    set_seed(SEED)  # same seed/budget as C15 for a fair comparison
    net = PolicyNetwork(obs_dim, n_actions)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    returns_hist = []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        log_probs, rewards = [], []
        for _ in range(500):
            obs_t = torch.as_tensor(np.asarray(obs, dtype=np.float32)).reshape(1, obs_dim)
            dist = Categorical(logits=net(obs_t))
            action = dist.sample()
            obs, reward, terminated, truncated, _ = env.step(int(action.item()))
            log_probs.append(dist.log_prob(action).squeeze())
            rewards.append(float(reward))
            if terminated or truncated:
                break
        returns_hist.append(float(np.sum(rewards)))
        G = compute_returns(np.asarray(rewards), gamma=gamma, mode='v2')
        # whitening only (no learned state-dependent baseline / critic)
        G = torch.as_tensor(normalize_advantages(G), dtype=torch.float32)
        log_probs_t = torch.stack(log_probs)
        loss = -(log_probs_t * G).sum()
        if torch.isfinite(loss):
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep + 1) % 50 == 0:
            print(f'REINFORCE ep {ep+1:>3}/{n_episodes}  return(avg last 20)={np.mean(returns_hist[-20:]):.1f}')
    return returns_hist

pg_returns = train_reinforce(env)

def replot(window: int = 10) -> None:
    plt.figure()
    plt.plot(moving_average(pg_returns, window), label='REINFORCE (no critic)')
    plt.plot(moving_average(a2c_returns, window), label='A2C (with critic)')
    plt.xlabel('episode'); plt.ylabel(f'return (moving avg, window={window})')
    plt.title('Learning curves: REINFORCE vs A2C')
    plt.legend(); plt.show()

if WIDGETS_AVAILABLE:
    widgets.interact(replot, window=widgets.IntSlider(min=1, max=50, step=1, value=10))
else:
    print('[ipywidgets unavailable — static overlay, window=10]')
    replot(10)

# --- self-checks ---
assert len(pg_returns) == N_EPISODES
assert len(moving_average(pg_returns, 10)) >= 1
replot(10)
print('C16 REINFORCE-vs-A2C comparison OK')

## Recap & synthesis

We walked one narrative spine from first principles to a working A2C agent:

1. **RL as `action = f(observation)`** — a policy network trained to maximize return, in a non-differentiable environment where actions must be *sampled* to explore.
2. **Policy gradient as weighted classification** `L = Σ A_n e_n` — and the **evolution of the weighting term**:
   - **V0** immediate reward → **V1** cumulated return → **V2** discounted return (tunable credit horizon via `γ`).
   - **V3** subtract a baseline & normalize → relative *advantages* with positive *and* negative signal.
   - **V4** TD advantage `A_t = r_t + γV(s_{t+1}) − V(s_t)` using a learned critic — lower variance.
3. **MC vs TD** — the bias/variance trade-off of value estimation; Sutton's 8-episode example showed TD exploiting transition structure (`V(A)=0.75`) that MC ignores (`V(A)=0`).
4. **A2C** — actor + critic sharing a trunk; its learning curve is **smoother and lower-variance** than baseline-free REINFORCE, the payoff of the whole progression.

### Where to go next (deferred topics)

- **PPO / off-policy methods** — reuse data safely with clipped objectives and importance sampling.
- **Exploration techniques** — entropy bonuses (we used a small one), count-based and curiosity-driven exploration.
- **Sparse-reward shaping** — when `Σ r_t` gives almost no gradient.
- **Imitation learning & inverse RL (IRL)** — learning from demonstrations or recovering the reward function itself.

Try bumping `N_EPISODES`, sliding `γ`, or toggling normalization to build intuition for how each ingredient changes the learning curve.